# 🚢 تجارت‌یار — اجرای سامانه روی Google Colab

این دفترچه سامانه **تجارت‌یار** (React + Express + Vite) را روی Colab نصب، build و اجرا می‌کند و یک **لینک عمومی موقت** می‌دهد.

**هر سلول را به ترتیب با `Ctrl+Enter` اجرا کنید** (یا از منوی `Runtime → Run all`):
1. نصب Node.js ۲۰
2. دریافت کد از GitHub
3. نصب وابستگی‌ها
4. build پروژه
5. اجرای سرور (UI + API روی پورت ۳۰۰۰)
6. بررسی سلامت
7. دانلود cloudflared
8. راه‌اندازی تونل (این سلول بلافاصله تمام می‌شود)
9. دریافت لینک دسترسی

> سلول‌های ۱ تا ۷ خودکار با `%%bash` و سلول‌های ۸ و ۹ با پایتون اجرا می‌شوند — فقط به ترتیب `Ctrl+Enter` بزنید، نیازی به تغییر چیزی نیست.
> لینک نهایی در خروجی **سلول ۹** چاپ می‌شود.

In [ ]:
%%bash
curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs && echo "Node: $(node -v) / npm: $(npm -v)"

In [ ]:
%%bash
cd /content && rm -rf Tejaratyarr && git clone --depth 1 --branch arena/01a04f8e-tejaratyarr https://github.com/Setayesh-Jafari/Tejaratyarr.git

In [ ]:
%%bash
cd /content/Tejaratyarr && npm install

In [ ]:
%%bash
cd /content/Tejaratyarr && npm run build

In [ ]:
%%bash
cd /content/Tejaratyarr && setsid nohup node dist/server.cjs > /content/server.log 2>&1 < /dev/null &

In [ ]:
%%bash
sleep 5; curl -s http://localhost:3000/api/health; echo

In [ ]:
%%bash
cd /content/Tejaratyarr && (test -x cloudflared || curl -L --progress-bar -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64) && chmod +x cloudflared && ./cloudflared --version

In [ ]:
import subprocess

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:3000",
     "--no-autoupdate", "--logfile", "/content/cloudflared.log"],
    cwd="/content/Tejaratyarr",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
print("cloudflared started (PID", p.pid, ") — tunnel is being created, go to the next cell.")


In [ ]:
import time, re

print("waiting for the tunnel URL (up to ~2 minutes)...")
url = None
for _ in range(60):
    try:
        with open("/content/cloudflared.log", "r", errors="ignore") as f:
            log = f.read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

if url:
    print()
    print("LINK:", url)
else:
    print()
    print("link not ready yet - last lines of cloudflared log:")
    try:
        with open("/content/cloudflared.log", "r", errors="ignore") as f:
            print(f.read()[-2000:])
    except Exception as e:
        print(e)


## 📌 نکات

- لینک `trycloudflare` **موقت** است و تا زمانی که Colab روشن بماند کار می‌کند.
- اگر لینک چاپ نشد: سلول ۸ را دوباره اجرا کنید، سپس سلول ۹ را اجرا کنید. اگر باز هم نشد، سلول ۶ را اجرا کنید تا مطمئن شوید سرور بالا است.
- داده‌ها در Colab موقتی است؛ برای استفاده‌ی واقعی روی سرور خودتان اجرا کنید.
- فعال‌سازی هوش مصنوعی Gemini: قبل از اجرای سرور (سلول ۵)، `GEMINI_API_KEY` را تنظیم کنید.
- اجرای محلی: `npm install` سپس `npm run dev` (پورت ۳۰۰۰).
- اجرای تولید: `npm run build` سپس `NODE_ENV=production node dist/server.cjs`.

## 🛠 رفع اشکال
- **خطای `SyntaxError` در سلول bash؟** یعنی خط اول سلول `%%bash` نیست.
- **سلول تونل کند است؟** در این نسخه تونل با پایتون راه‌اندازی می‌شود و بلافاصله تمام می‌شود؛ منتظر ماندن فقط مربوط به سلول ۹ (دریافت لینک) است.